# Data Exploration: Processed Credit Risk Datasets

This notebook:
1. **Preprocesses** all raw datasets if the processed folder is empty (using `src.data.preprocessing`)
2. **Explores** the processed datasets with statistics and visualizations

Tasks covered:
- **PD (Probability of Default)**: Binary classification
- **LGD (Loss Given Default)**: Regression

In [ ]:
import sys
from pathlib import Path

# Add project root to path (notebook is in notebooks/ folder)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plot settings
plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['figure.dpi'] = 100

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# =============================================================================
# IMPORT PREPROCESSING FROM src/data/
# =============================================================================

from src.data.preprocessing import preprocess_dataset


# Define directories
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'

print(f"Raw data dir: {RAW_DIR} (exists: {RAW_DIR.exists()})")
print(f"Processed dir: {PROCESSED_DIR} (exists: {PROCESSED_DIR.exists()})")

In [ ]:
# =============================================================================
# GET AVAILABLE DATASETS FROM RAW FOLDER
# =============================================================================

def get_available_datasets(raw_dir: Path) -> dict:
    """Get list of available raw datasets (CSV and Parquet files)."""
    datasets = {'pd': [], 'lgd': []}
    
    if (raw_dir / 'pd').exists():
        csv_files = {f.stem for f in (raw_dir / 'pd').glob('*.csv')}
        parquet_files = {f.stem for f in (raw_dir / 'pd').glob('*.parquet')}
        datasets['pd'] = sorted(csv_files | parquet_files)  # Union of both
    
    if (raw_dir / 'lgd').exists():
        csv_files = {f.stem for f in (raw_dir / 'lgd').glob('*.csv')}
        parquet_files = {f.stem for f in (raw_dir / 'lgd').glob('*.parquet')}
        datasets['lgd'] = sorted(csv_files | parquet_files)  # Union of both
    
    return datasets

available_datasets = get_available_datasets(RAW_DIR)
print(f"Available PD datasets: {len(available_datasets['pd'])}")
for d in available_datasets['pd']:
    print(f"  - {d}")
print(f"\nAvailable LGD datasets: {len(available_datasets['lgd'])}")
for d in available_datasets['lgd']:
    print(f"  - {d}")

---
## Part 1: Preprocess All Datasets (if needed)

In [ ]:
# =============================================================================
# CHECK IF PREPROCESSING IS NEEDED
# =============================================================================

def check_processed_exists(processed_dir: Path) -> dict:
    """Check which datasets are already processed."""
    processed = {'pd': [], 'lgd': []}
    
    pd_dir = processed_dir / 'pd'
    lgd_dir = processed_dir / 'lgd'
    
    if pd_dir.exists():
        # Check for dataset folders with y.npy files
        for d in pd_dir.iterdir():
            if d.is_dir() and (d / 'y.npy').exists():
                processed['pd'].append(d.name)
    
    if lgd_dir.exists():
        for d in lgd_dir.iterdir():
            if d.is_dir() and (d / 'y.npy').exists():
                processed['lgd'].append(d.name)
    
    return processed

processed_datasets = check_processed_exists(PROCESSED_DIR)
print(f"Already processed PD: {len(processed_datasets['pd'])}")
print(f"Already processed LGD: {len(processed_datasets['lgd'])}")

In [ ]:
# =============================================================================
# PREPROCESS ALL DATASETS
# =============================================================================

def preprocess_all_datasets(available: dict, processed: dict):
    """Preprocess all datasets that haven't been processed yet."""
    
    # PD datasets
    pd_to_process = [d for d in available['pd'] if d not in processed['pd']]
    if pd_to_process:
        print("=" * 60)
        print(f" PREPROCESSING {len(pd_to_process)} PD DATASETS")
        print("=" * 60)
        for dataset in pd_to_process:
            try:
                print(f"\nProcessing: {dataset}")
                N, C, y, info = preprocess_dataset('pd', dataset)
                print(f"  Done: {info['n_samples']} samples, {info['n_num_features']} num, {info['n_cat_features']} cat")
            except Exception as e:
                print(f"  ERROR: {e}")
    else:
        print("All PD datasets already processed.")
    
    # LGD datasets
    lgd_to_process = [d for d in available['lgd'] if d not in processed['lgd']]
    if lgd_to_process:
        print("\n" + "=" * 60)
        print(f" PREPROCESSING {len(lgd_to_process)} LGD DATASETS")
        print("=" * 60)
        for dataset in lgd_to_process:
            try:
                print(f"\nProcessing: {dataset}")
                N, C, y, info = preprocess_dataset('lgd', dataset)
                print(f"  Done: {info['n_samples']} samples, {info['n_num_features']} num, {info['n_cat_features']} cat")
            except Exception as e:
                print(f"  ERROR: {e}")
    else:
        print("All LGD datasets already processed.")

# Run preprocessing
preprocess_all_datasets(available_datasets, processed_datasets)

# Update processed list
processed_datasets = check_processed_exists(PROCESSED_DIR)
print(f"\nTotal processed - PD: {len(processed_datasets['pd'])}, LGD: {len(processed_datasets['lgd'])}")

---
## Part 2: Data Exploration Functions

In [ ]:
# =============================================================================
# EXPLORATION HELPER FUNCTIONS (Using DataFeeder - Training Set Only)
# =============================================================================

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from src.data.data_feeder import DataFeeder
from src.data.preprocessing import preprocess_dataset

PROJECT_ROOT = Path(__file__).resolve().parents[2] if '__file__' in globals() else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"


def load_training_data_with_names(
    task: str, 
    dataset: str,
    test_size: float = 0.2,
    val_size: float = 0.15,
    seed: int = 42,
    row_limit: int = None,
    sampling: float = None,
    apply_pca: bool = True,
    remove_outliers: bool = True
) -> tuple:
    """
    Load TRAINING SET ONLY with original feature names preserved.
    
    This shows exactly what the model sees during training, with proper feature names.
    
    Parameters
    ----------
    task : str
        'pd' or 'lgd'
    dataset : str
        Dataset name (e.g., '0001.gmsc')
    apply_pca : bool
        Whether to apply PCA (should match your experiment settings)
    remove_outliers : bool
        Whether to remove outliers (should match your experiment settings)
    
    Returns
    -------
    N : np.ndarray or None
        Numerical features from TRAINING set
    C : np.ndarray or None
        Categorical features from TRAINING set
    y : np.ndarray
        Target variable from TRAINING set
    feature_names : dict
        {'numerical': [...], 'categorical': [...]}
    info : dict
        Metadata about the dataset
    """
    try:
        print(f"Loading {dataset} training data with DataFeeder...")
        print(f"  Settings: PCA={apply_pca}, Outliers={remove_outliers}")
        
        # Step 1: Load RAW data to get original feature names
        N_raw, C_raw, y_raw, info_raw = preprocess_dataset(task, dataset)
        original_num_cols = info_raw.get('numerical_cols', [f'num_{i}' for i in range(N_raw.shape[1] if N_raw is not None else 0)])
        original_cat_cols = info_raw.get('categorical_cols', [f'cat_{i}' for i in range(C_raw.shape[1] if C_raw is not None else 0)])
        
        print(f"  Original: {len(original_num_cols)} numerical, {len(original_cat_cols)} categorical features")
        
        # Step 2: Apply DataFeeder preprocessing
        feeder = DataFeeder(
            task=task,
            dataset=dataset,
            test_size=test_size,
            val_size=val_size,
            cv_splits=1,  # Single fold
            seed=seed,
            row_limit=row_limit,
            sampling=sampling,
            apply_pca=apply_pca,
            remove_outliers=remove_outliers
        )
        
        folds = feeder.prepare()
        (N_dict, C_dict, y_dict), info = folds[1]
        
        # Step 3: Extract TRAINING set only
        N_train = N_dict['train'] if N_dict is not None else None
        C_train = C_dict['train'] if C_dict is not None else None
        y_train = y_dict['train']
        
        print(f"  Training set: {len(y_train)} samples")
        print(f"  After preprocessing: {info['n_num_features']} numerical, {info['n_cat_features']} categorical features")
        
        # Step 4: Determine feature names based on what preprocessing was applied
        feature_names = {'numerical': [], 'categorical': []}
        pca_was_applied = False
        
        # Check if PCA was applied by comparing feature counts and categorical disappearance
        original_total = len(original_num_cols) + len(original_cat_cols)
        current_total = info['n_num_features'] + info['n_cat_features']
        
        if apply_pca and original_total > 100:
            # PCA should have been applied
            if info['n_cat_features'] == 0 and info['n_num_features'] <= 99:
                pca_was_applied = True
                feature_names['numerical'] = [f'PC{i+1}' for i in range(info['n_num_features'])]
                print(f"  ✓ PCA was applied: {original_total} features → {info['n_num_features']} components")
            else:
                print(f"  ✗ PCA was NOT applied (total features: {original_total} ≤ 100)")
        else:
            print(f"  ✗ PCA was NOT applied (total features: {original_total} ≤ 100)")
        
        # If PCA was NOT applied, we need to figure out which original columns were kept
        if not pca_was_applied:
            # For numerical features: some may have been dropped (near-constant)
            if N_train is not None and len(original_num_cols) > 0:
                if N_train.shape[1] == len(original_num_cols):
                    # All columns kept
                    feature_names['numerical'] = original_num_cols
                else:
                    # Some columns were dropped - we can't easily track which ones without more info
                    # So we use generic names but indicate they're a subset
                    feature_names['numerical'] = [f'{original_num_cols[0][:10]}..._subset_{i}' 
                                                 for i in range(N_train.shape[1])]
                    print(f"  ⚠ {len(original_num_cols) - N_train.shape[1]} numerical features were dropped (near-constant)")
            
            # For categorical features
            if C_train is not None and len(original_cat_cols) > 0:
                if C_train.shape[1] == len(original_cat_cols):
                    # All columns kept
                    feature_names['categorical'] = original_cat_cols
                else:
                    # Some columns were dropped
                    feature_names['categorical'] = [f'{original_cat_cols[0][:10]}..._subset_{i}' 
                                                   for i in range(C_train.shape[1])]
                    print(f"  ⚠ {len(original_cat_cols) - C_train.shape[1]} categorical features were dropped (near-constant)")
        
        # Build comprehensive info dict
        info_complete = {
            'task_type': info['task_type'],
            'n_samples': len(y_train),
            'n_num_features': info['n_num_features'],
            'n_cat_features': info['n_cat_features'],
            'split_used': 'train',
            'preprocessing_applied': {
                'pca': pca_was_applied,
                'outlier_removal': remove_outliers,
                'constant_columns_removed': True
            },
            'original_num_features': len(original_num_cols),
            'original_cat_features': len(original_cat_cols),
        }
        
        return N_train, C_train, y_train, feature_names, info_complete
        
    except Exception as e:
        print(f"✗ Error loading dataset: {e}")
        import traceback
        traceback.print_exc()
        return None, None, None, None, None


def plot_target_distribution(y: np.ndarray, task: str, dataset_name: str):
    """Plot target variable distribution."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    if task == 'pd':
        unique, counts = np.unique(y, return_counts=True)
        axes[0].bar([str(int(u)) for u in unique], counts, color=['steelblue', 'coral'], edgecolor='black')
        axes[0].set_xlabel('Class')
        axes[0].set_ylabel('Count')
        axes[0].set_title('Class Distribution')
        for i, (u, c) in enumerate(zip(unique, counts)):
            axes[0].annotate(f'{100*c/len(y):.1f}%', xy=(i, c), ha='center', va='bottom')
        axes[1].pie(counts, labels=[f'Class {int(u)}' for u in unique], autopct='%1.1f%%', colors=['steelblue', 'coral'])
        axes[1].set_title('Class Proportion')
    else:
        axes[0].hist(y, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
        axes[0].axvline(x=np.mean(y), color='red', linestyle='--', label=f'Mean: {np.mean(y):.3f}')
        axes[0].axvline(x=np.median(y), color='green', linestyle='--', label=f'Median: {np.median(y):.3f}')
        axes[0].set_xlabel('LGD')
        axes[0].set_ylabel('Frequency')
        axes[0].set_title('Target Distribution')
        axes[0].legend()
        axes[1].boxplot(y, vert=True)
        axes[1].set_ylabel('LGD')
        axes[1].set_title('Target Box Plot')
    
    plt.suptitle(f'{dataset_name} - Training Set', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


def plot_feature_distributions(N: np.ndarray, feature_names: list, dataset_name: str):
    """Plot numeric feature distributions for ALL features with ORIGINAL names."""
    if N is None or N.shape[1] == 0:
        print("No numeric features to plot.")
        return
    
    n_features = N.shape[1]
    n_cols = 4
    n_rows = (n_features + n_cols - 1) // n_cols
    
    # Adjust figure size based on number of rows
    fig_height = min(3 * n_rows, 100)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, fig_height))
    
    # Handle single row case
    if n_rows == 1:
        axes = axes.reshape(1, -1)
    axes = axes.flatten()
    
    # Plot all features with ORIGINAL names
    for i in range(n_features):
        col_name = feature_names[i] if i < len(feature_names) else f'feature_{i}'
        data = N[:, i]
        data = data[~np.isnan(data)]
        
        axes[i].hist(data, bins=30, edgecolor='black', alpha=0.7, color='steelblue')
        
        # Truncate long names for display
        display_name = col_name if len(col_name) <= 25 else col_name[:22] + '...'
        axes[i].set_title(display_name, fontsize=10)
        axes[i].tick_params(labelsize=8)
        
        # Add statistics
        axes[i].text(0.02, 0.98, f'n={len(data)}', transform=axes[i].transAxes,
                     fontsize=7, verticalalignment='top',
                     bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))
    
    # Hide unused subplots
    for i in range(n_features, len(axes)):
        axes[i].set_visible(False)
    
    plt.suptitle(f'{dataset_name} - Training Set - All {n_features} Feature Distributions', 
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print(f"Displayed all {n_features} numeric features with original names")


def plot_correlation_heatmap(N: np.ndarray, y: np.ndarray, feature_names: list, dataset_name: str):
    """Plot correlation heatmap with ORIGINAL feature names."""
    if N is None or N.shape[1] < 2:
        print("Not enough numeric features for correlation.")
        return
    
    # Combine N and y for correlation
    data = np.column_stack([N, y])
    
    # Handle NaN
    df = pd.DataFrame(data)
    corr_matrix = df.corr()
    
    # Get top correlated with target
    target_corr = corr_matrix.iloc[:-1, -1].abs().sort_values(ascending=False)
    top_indices = target_corr.head(14).index.tolist()
    top_indices.append(len(corr_matrix) - 1)  # Add target
    
    sub_corr = corr_matrix.iloc[top_indices, top_indices]
    
    # Create labels using ORIGINAL names
    labels = []
    for idx in top_indices[:-1]:
        name = feature_names[idx] if idx < len(feature_names) else f'feature_{idx}'
        # Truncate long names
        labels.append(name if len(name) <= 15 else name[:12] + '...')
    labels.append('target')
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(sub_corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
                xticklabels=labels, yticklabels=labels,
                square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
    plt.title(f'{dataset_name} - Training Set Correlations', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


def explore_dataset(
    task: str, 
    dataset: str,
    apply_pca: bool = True,
    remove_outliers: bool = True,
    row_limit: int = None
) -> dict:
    """
    Comprehensive exploration of TRAINING SET with original feature names.
    
    This shows exactly what your models are trained on, with proper feature names.
    
    Parameters
    ----------
    task : str
        'pd' or 'lgd'
    dataset : str
        Dataset name (e.g., '0001.gmsc')
    apply_pca : bool
        Whether to apply PCA (should match your experiment settings)
    remove_outliers : bool
        Whether to remove outliers (should match your experiment settings)
    row_limit : int, optional
        Limit number of rows for faster exploration
    
    Returns
    -------
    dict : Summary statistics
    """
    print("\n" + "=" * 80)
    print(f" DATASET: {dataset} ({task.upper()}) - TRAINING SET EXPLORATION")
    print("=" * 80)
    
    N, C, y, feature_names, info = load_training_data_with_names(
        task=task,
        dataset=dataset,
        apply_pca=apply_pca,
        remove_outliers=remove_outliers,
        row_limit=row_limit
    )
    
    if y is None:
        print("Dataset not found!")
        return None
    
    # Basic statistics
    print(f"\n[BASIC STATISTICS]")
    print(f"  Split:            {info['split_used'].upper()}")
    print(f"  Samples:          {info['n_samples']:,}")
    print(f"  Numeric features: {info['n_num_features']} (original: {info['original_num_features']})")
    print(f"  Categorical feat: {info['n_cat_features']} (original: {info['original_cat_features']})")
    print(f"  Total features:   {info['n_num_features'] + info['n_cat_features']}")
    
    # Preprocessing info
    print(f"\n[PREPROCESSING APPLIED]")
    pp = info.get('preprocessing_applied', {})
    print(f"  PCA:              {pp.get('pca', False)}")
    print(f"  Outlier removal:  {pp.get('outlier_removal', False)}")
    print(f"  Constant cols:    {pp.get('constant_columns_removed', False)}")
    
    # Feature names sample
    print(f"\n[FEATURE NAMES]")
    if feature_names['numerical']:
        print(f"  Numerical features (first 5): {feature_names['numerical'][:5]}")
        if len(feature_names['numerical']) > 5:
            print(f"                    ... and {len(feature_names['numerical']) - 5} more")
    if feature_names['categorical']:
        print(f"  Categorical features (first 5): {feature_names['categorical'][:5]}")
        if len(feature_names['categorical']) > 5:
            print(f"                      ... and {len(feature_names['categorical']) - 5} more")
    
    # Target statistics
    print(f"\n[TARGET VARIABLE]")
    if task == 'pd':
        unique, counts = np.unique(y, return_counts=True)
        for u, c in zip(unique, counts):
            print(f"  Class {int(u)}: {c:,} ({100*c/len(y):.1f}%)")
        print(f"  Imbalance ratio: {counts.max()/counts.min():.2f}:1")
    else:
        print(f"  Min: {y.min():.4f}  Max: {y.max():.4f}")
        print(f"  Mean: {y.mean():.4f}  Median: {np.median(y):.4f}  Std: {y.std():.4f}")
        n_zero = (y == 0).sum()
        n_one = (y == 1).sum()
        print(f"  Values=0: {n_zero} ({100*n_zero/len(y):.1f}%)  Values=1: {n_one} ({100*n_one/len(y):.1f}%)")
    
    # Visualizations
    plot_target_distribution(y, task, dataset)
    
    if N is not None:
        plot_feature_distributions(N, feature_names['numerical'], dataset)
        plot_correlation_heatmap(N, y, feature_names['numerical'], dataset)
    
    return {
        'name': dataset,
        'task': task,
        'split': 'train',
        'n_samples': info['n_samples'],
        'n_num_features': info['n_num_features'],
        'n_cat_features': info['n_cat_features'],
        'n_total_features': info['n_num_features'] + info['n_cat_features'],
        'preprocessing': pp,
        'feature_names': feature_names
    }


# =============================================================================
# CONVENIENCE FUNCTIONS
# =============================================================================

def compare_preprocessing_effects(task: str, dataset: str, row_limit: int = None):
    """
    Compare the TRAINING SET with and without preprocessing.
    
    Shows the effect of PCA and outlier removal on training data.
    """
    print("\n" + "=" * 80)
    print(f" PREPROCESSING COMPARISON: {dataset} ({task.upper()}) - TRAINING SET")
    print("=" * 80)
    
    print("\n[1] WITHOUT PREPROCESSING (PCA=False, outliers=False)")
    print("-" * 80)
    results_no_pp = explore_dataset(
        task=task,
        dataset=dataset,
        apply_pca=False,
        remove_outliers=False,
        row_limit=row_limit
    )
    
    print("\n\n[2] WITH PREPROCESSING (PCA=True, outliers=True)")
    print("-" * 80)
    results_with_pp = explore_dataset(
        task=task,
        dataset=dataset,
        apply_pca=True,
        remove_outliers=True,
        row_limit=row_limit
    )
    
    # Summary comparison
    print("\n" + "=" * 80)
    print(" SUMMARY COMPARISON (TRAINING SET)")
    print("=" * 80)
    if results_no_pp and results_with_pp:
        print(f"  Samples: {results_no_pp['n_samples']:,} → {results_with_pp['n_samples']:,} "
              f"({results_with_pp['n_samples'] - results_no_pp['n_samples']:+,})")
        print(f"  Features: {results_no_pp['n_total_features']} → {results_with_pp['n_total_features']} "
              f"({results_with_pp['n_total_features'] - results_no_pp['n_total_features']:+})")
        
        if results_with_pp['preprocessing']['pca']:
            print(f"  ✓ PCA was applied: {results_no_pp['n_total_features']} features → {results_with_pp['n_total_features']} components")
        else:
            print(f"  ✗ PCA was NOT applied (total features ≤ 100)")


def quick_explore(task: str, dataset: str):
    """
    Quick exploration with default settings (PCA=True, outliers=True).
    
    This shows what the model actually sees during training.
    """
    return explore_dataset(task, dataset, apply_pca=True, remove_outliers=True)

---
## Part 3: Summary Comparison

In [ ]:
# =============================================================================
# SUMMARY TABLE (WITH PREPROCESSING INFO)
# =============================================================================

print("=" * 80)
print(" DATASET SUMMARY TABLE")
print("=" * 80)

all_summaries = pd_summaries + lgd_summaries

if all_summaries:
    # Extract relevant info from each summary
    table_data = []
    for s in all_summaries:
        pp = s.get('preprocessing', {})
        table_data.append({
            'Dataset': s['name'],
            'Task': s['task'].upper(),
            'Samples': s['n_samples'],
            'Numeric': s['n_num_features'],
            'Categorical': s['n_cat_features'],
            'Total': s['n_total_features'],
            'PCA': '✓' if pp.get('pca', False) else '✗',
            'Outliers': '✓' if pp.get('outlier_removal', False) else '✗'
        })
    
    summary_df = pd.DataFrame(table_data)
    
    print("\n")
    print(summary_df.to_string(index=False))
    print("\nNote: PCA/Outliers columns show what preprocessing was applied")
else:
    print("No datasets loaded.")

In [ ]:
# =============================================================================
# VISUALIZATION: DATASET COMPARISON
# =============================================================================

if all_summaries:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    names = [s['name'][:15] for s in all_summaries]
    samples = [s['n_samples'] for s in all_summaries]
    features = [s['n_total_features'] for s in all_summaries]
    tasks = [s['task'] for s in all_summaries]
    colors = ['steelblue' if t == 'pd' else 'coral' for t in tasks]
    
    # Samples
    bars1 = axes[0].barh(names, samples, color=colors, edgecolor='black')
    axes[0].set_xlabel('Number of Samples')
    axes[0].set_title('Dataset Size')
    axes[0].invert_yaxis()
    for bar, val in zip(bars1, samples):
        axes[0].text(val, bar.get_y() + bar.get_height()/2, f' {val:,}', va='center', fontsize=9)
    
    # Features
    bars2 = axes[1].barh(names, features, color=colors, edgecolor='black')
    axes[1].set_xlabel('Number of Features')
    axes[1].set_title('Dimensionality')
    axes[1].invert_yaxis()
    for bar, val in zip(bars2, features):
        axes[1].text(val, bar.get_y() + bar.get_height()/2, f' {val}', va='center', fontsize=9)
    
    # Feature types
    n_num = [s['n_num_features'] for s in all_summaries]
    n_cat = [s['n_cat_features'] for s in all_summaries]
    y_pos = np.arange(len(names))
    axes[2].barh(y_pos, n_num, label='Numeric', color='steelblue', edgecolor='black')
    axes[2].barh(y_pos, n_cat, left=n_num, label='Categorical', color='coral', edgecolor='black')
    axes[2].set_yticks(y_pos)
    axes[2].set_yticklabels(names)
    axes[2].set_xlabel('Features')
    axes[2].set_title('Feature Types')
    axes[2].legend(loc='lower right')
    axes[2].invert_yaxis()
    
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor='steelblue', label='PD'), Patch(facecolor='coral', label='LGD')]
    fig.legend(handles=legend_elements, loc='upper right', title='Task')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# =============================================================================
# TASK-SPECIFIC STATISTICS
# =============================================================================

print("=" * 80)
print(" TASK-SPECIFIC SUMMARY")
print("=" * 80)

if pd_summaries:
    print("\n[PD DATASETS (Classification)]")
    print(f"  Total datasets:       {len(pd_summaries)}")
    print(f"  Total samples:        {sum(s['n_samples'] for s in pd_summaries):,}")
    print(f"  Avg samples/dataset:  {np.mean([s['n_samples'] for s in pd_summaries]):,.0f}")
    print(f"  Avg features/dataset: {np.mean([s['n_total_features'] for s in pd_summaries]):.1f}")

if lgd_summaries:
    print("\n[LGD DATASETS (Regression)]")
    print(f"  Total datasets:       {len(lgd_summaries)}")
    print(f"  Total samples:        {sum(s['n_samples'] for s in lgd_summaries):,}")
    print(f"  Avg samples/dataset:  {np.mean([s['n_samples'] for s in lgd_summaries]):,.0f}")
    print(f"  Avg features/dataset: {np.mean([s['n_total_features'] for s in lgd_summaries]):.1f}")

---
## Part 4: Explore PD Datasets

In [ ]:
# =============================================================================
# EXPLORE ALL PD DATASETS
# =============================================================================

pd_summaries = []

print(f"Exploring {len(processed_datasets['pd'])} PD datasets...")

for dataset in sorted(processed_datasets['pd']):
    summary = explore_dataset('pd', dataset)
    if summary:
        pd_summaries.append(summary)

---
## Part 5: Explore LGD Datasets

In [ ]:
# =============================================================================
# EXPLORE ALL LGD DATASETS
# =============================================================================

lgd_summaries = []

print(f"Exploring {len(processed_datasets['lgd'])} LGD datasets...")

for dataset in sorted(processed_datasets['lgd']):
    summary = explore_dataset('lgd', dataset)
    if summary:
        lgd_summaries.append(summary)